<a href="https://colab.research.google.com/github/Aryan2079/LLM_finetuning/blob/main/llm_finetuning_for_professional_laguage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U peft transformers bitsandbytes accelerate trl

In [ ]:
from datasets import load_dataset, IterableDataset

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
import re

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"
dataset_name = "Brianferrell787/financial-news-multisource"
chunk_size=512


In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
dataset = load_dataset(
    dataset_name,
    split="train",
    streaming=True,
    token=HF_TOKEN
)
model = AutoModelForCausalLM(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/131 [00:00<?, ?it/s]

In [ ]:
def clean_text(text):
  chunks = re.split(r'\n\s*\n', text)
  clean_text = [chunk.strip() for chunk in chunks[1:]]
  return clean_text

In [ ]:
def chunk_tokens(tokens, chunk_size=chunk_size):
  chunks = []
  for i in range(0, len(tokens), chunk_size):
    chunks.append(tokens[i:i+chunk_size])

  return chunks

In [ ]:
def clean_chunk_tokenize(dataset):
  for row in dataset:
    text = clean_text(row['text'])

    for paragraph in text:
      tokens = tokenizer(paragraph, truncation=False)["input_ids"]

      for chunk in chunk_tokens(tokens, chunk_size):
        yield {"input_ids": chunk, "labels":chunk.copy()}

In [ ]:
finetune_dataset = IterableDataset.from_generator(
    lambda: clean_chunk_tokenize(dataset)
)

In [ ]:
q_lora_model = get_peft_model(model, lora_config)

In [ ]:
trainer = Trainer(
    model=q_lora_model,
    args=training_args,
    train_dataset=finetune_dataset
)

In [ ]:
trainer.train()